<a href="https://colab.research.google.com/github/lmassaron/ml4dummies_3ed/blob/main/ML4D3E_16_classifying_images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q datasets

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import keras
from keras import layers, optimizers
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
ds = load_dataset("lmassaron/dogs-cats-openimages")

In [ ]:
print(f"Train set size: {len(ds['train'])}")
print(f"Validation set size: {len(ds['validation'])}")
print(f"Test set size: {len(ds['test'])}")

In [ ]:
IMAGE_WIDTH = 256
IMAGE_HEIGHT = 256
BATCH_SIZE = 32
EPOCHS = 25

In [ ]:
def process_images(data):
  all_images_list = []
  all_labels_list = []

  for example in tqdm(data):
    pil_image = example['image']
    label = example['label']
    img_rgb = pil_image.convert("RGB")
    img_resized = img_rgb.resize((IMAGE_WIDTH,
                                  IMAGE_HEIGHT))
    img_array = keras.utils.img_to_array(img_resized,
                    dtype='float32')

    all_images_list.append(img_array)
    all_labels_list.append(label)

  x_np = np.array(all_images_list)
  y_np = np.array(all_labels_list)
  return x_np, y_np

x_train, y_train = process_images(ds['train'])
x_valid, y_valid = process_images(ds['validation'])
x_test, y_test = process_images(ds['test'])

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=(-0.05, 0.05) )
    ],
    name="data_augmentation")

In [ ]:
def conv_block(filters, kernel_size, name):
  params = {
      "filters": filters,
      "kernel_size": kernel_size,
      "activation": "relu",
      "padding": "same"}
  block = [
      layers.Conv2D(**params),
      layers.Conv2D(**params),
      layers.MaxPooling2D((2, 2))
  ]
  return keras.Sequential(block, name=name)

In [ ]:
keras.utils.set_random_seed(0)

model = keras.Sequential(
    [
        keras.Input(shape=(IMAGE_HEIGHT,
                           IMAGE_WIDTH, 3)),
        layers.Rescaling(1.0 / 255),
        data_augmentation,

        conv_block(32, 5, name="conv_block1"),
        layers.BatchNormalization(),
        conv_block(32, 4, name="conv_block2"),
        conv_block(64, 4, name="conv_block3"),
        layers.BatchNormalization(),
        conv_block(64, 3, name="conv_block4"),
        conv_block(128, 3, name="conv_block5"),
        layers.BatchNormalization(),
        conv_block(128, 2, name="conv_block6"),
        conv_block(256, 2, name="conv_block7"),
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),

        layers.Dropout(0.2),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="simple_cat_dog_classifier")

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=optimizers.Adam(),
    loss = "binary_crossentropy",
    metrics = ["accuracy"])

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True)

history = model.fit(
    x_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_valid, y_valid),
    shuffle=True,
    callbacks=[early_stopping])

In [ ]:
def plot_history(history):
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.title("model accuracy")
    plt.ylabel("accuracy")
    plt.xlabel("epoch")
    plt.legend(["train", "validation"],
               loc="upper left")
    plt.show()

plot_history(history)

In [ ]:
loss_value, accuracy_value = model.evaluate(
    x_test, y_test, batch_size=BATCH_SIZE)
print(f"test accuracy {accuracy_value:0.3f}")

In [ ]:
keras.utils.set_random_seed(0)

transfer_model = keras.applications.EfficientNetV2B0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3))

transfer_model.trainable = False

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=(
                  IMAGE_HEIGHT, IMAGE_WIDTH, 3)),
        data_augmentation,
        transfer_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ], name="EffNetV2_cat_dog_classifier")

In [ ]:
model.compile(
    optimizer=optimizers.Adam(),
    loss = "binary_crossentropy",
    metrics = ["accuracy"])

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=1,
    validation_data=(x_valid, y_valid),
    shuffle=True)

In [ ]:
loss_value, accuracy_value = model.evaluate(
    x_test,
    y_test,
    batch_size=BATCH_SIZE)
print(f"test accuracy {accuracy_value:0.3f}")